https://www.kaggle.com/code/ahsan442002/hbo-shows-movies-note-book?select=HBO_Dataset.csv

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
df = pd.read_csv('HBO_Dataset.csv', encoding="latin1")
df.to_csv("dados_utf8.csv", index=False, encoding="utf-8")

In [ ]:
df = pd.read_csv('dados_utf8.csv')

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
#df = df.dropna()

In [ ]:
unique_values_count = []
for i in list(df.columns[:]):
    print(f"{i}: {len(df[i].astype(str).value_counts())}")
    unique_values_count.append(len(df[i].astype(str).value_counts()))

In [ ]:
df.describe()

In [ ]:
#df.groupby(['Title']).size()

In [ ]:
df.groupby(['Type']).size()

In [ ]:
ax = df['Type'].value_counts().plot(
    kind='bar',
    color=plt.cm.Set2.colors,
    edgecolor='black',
    title='Type'
)

ax.set_xlabel('Type')
ax.set_ylabel('Count')
ax.bar_label(ax.containers[0])

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))

ax = sns.countplot(
    data=df,
    x='Type',
    hue='Type',
    palette='Set2',
    legend=False
)

ax.set_title('Type')
ax.set_xlabel('Type')
ax.set_ylabel('Count')

ax.bar_label(ax.containers[0])

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))

ax = sns.countplot(
    data=df,
    y='Type',
    hue='Type',
    palette='Set2',
    legend=False,
    order=df['Type'].value_counts().index
)

ax.set_title('Type')
ax.set_xlabel('Count')
ax.set_ylabel('Type')

for container in ax.containers:
    ax.bar_label(container, padding=3)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
df.groupby(['Release Year']).size()

In [ ]:
plt.figure(figsize=(6, 4))

ax = sns.countplot(
    data=df,
    y='Release Year',
    hue='Release Year',
    palette='Set2',
    legend=False,
    order=df['Release Year'].value_counts().index
)

ax.set_title('Release Year')
ax.set_xlabel('Count')
ax.set_ylabel('Release Year')

for container in ax.containers:
    ax.bar_label(container, padding=3)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
df.groupby(['Platform']).size()

In [ ]:
df.groupby(['Genre']).size()

In [ ]:
df.groupby(['Creator/Director']).size()

In [ ]:
df.groupby(['Main Cast']).size()

In [ ]:
df.groupby(['IMDb']).size()

In [ ]:
df.groupby(['RT Score']).size()

In [ ]:
df.groupby(['Awards & Noms']).size()

In [ ]:
df.groupby(['Seasons']).size()

In [ ]:
plt.figure(figsize=(6, 4))

ax = sns.countplot(
    data=df,
    y='Seasons',
    hue='Seasons',
    palette='Set2',
    legend=False,
    order=df['Seasons'].value_counts().index
)

ax.set_title('Seasons')
ax.set_xlabel('Count')
ax.set_ylabel('Seasons')

for container in ax.containers:
    ax.bar_label(container, padding=3)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
df.groupby(['Short Synopsis']).size()

In [ ]:
df.groupby(['Viewership/Views']).size()

In [ ]:
df.groupby(['Popularity Indicator']).size()

In [ ]:
df_copy = df.copy()

In [ ]:
df_copy['RT Score'] = (
    df_copy['RT Score']
    .str.rstrip('%')
    .astype(float))
)

In [ ]:
df_copy['RT Score'] = df_copy['RT Score']/100
df_copy['RT Score']

In [ ]:
df_copy['RT Score']

In [ ]:
corr = df_copy[['IMDb', 'RT Score']].corr()
print(corr)

In [ ]:
df_copy['IMDb'] = df_copy['IMDb']/10

In [ ]:
plt.figure(figsize=(6,4))

sns.regplot(
    data=df_copy,
    x='IMDb',
    y='RT Score',
    scatter_kws={'alpha':0.7},
    line_kws={'color':'red'}
)

plt.title('IMDb × Rotten Tomatoes')
plt.tight_layout()
plt.show()

In [ ]:
numerical_vars = []
for i in list(df.columns[:]):
    if df.dtypes[i] == 'int64' or df.dtypes[i] == 'float64':
        numerical_vars.append(i)
numerical_vars

In [ ]:
#fig, axes = plt.subplots(len(numerical_vars), 1, figsize=(8, 10))
fig, axes = plt.subplots(1, len(numerical_vars), figsize=(14, 4))

for ax, col in zip(axes, numerical_vars):
    sns.boxplot(data=df, y=col, ax=ax, color='skyblue')
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(numerical_vars), figsize=(14, 4))

for ax, col in zip(axes, numerical_vars):
    sns.histplot(data=df, y=col, ax=ax, color='skyblue')
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(numerical_vars), figsize=(14, 4))

for ax, col in zip(axes, numerical_vars):
    sns.histplot(data=df, x=col, ax=ax, color='skyblue')
    ax.set_title(col)

plt.tight_layout()
plt.show()

2. DATA CLEANING & PREPROCESSING PIPELINE

In [ ]:
def parse_viewership_to_millions(val):
    """
    Standardizes disparate view configurations into normalized millions (float).
    Handles standard millions ('M'), billion-minute indexes ('B'), and daily counts.
    """
    if pd.isna(val):
        return np.nan
    
    val_str = str(val).strip().upper()
    
    try:
        # Case A: Standardized Millions (e.g., 20.2M avg, 12M total)
        if 'M' in val_str:
            match = re.search(r"([0-9\.]+)\s*M", val_str)
            if match:
                return float(match.group(1))
                
        # Case B: Modern Streaming Minutes (e.g., 11.9B min)
        # Scaled down using an industry-standard relative viewership weight
        elif 'B' in val_str:
            match = re.search(r"([0-9\.]+)\s*B", val_str)
            if match:
                return float(match.group(1)) * 15.0
                
        # Case C: Micro-scale Daily Trackers (e.g., 194 (Daily))
        elif 'DAILY' in val_str or '(' in val_str:
            match = re.search(r"^([0-9\.]+)", val_str)
            if match:
                return float(match.group(1)) / 100.0
                
        # Case D: Pure Numerical Fallback
        match = re.search(r"([0-9\.]+)", val_str)
        if match:
            return float(match.group(1))
            
    except Exception as e:
        return np.nan
        
    return np.nan

In [ ]:
df['Viewership_Millions'] = df['Viewership/Views'].apply(parse_viewership_to_millions)
df

In [ ]:
# %pip install numpy==1.26.4

In [ ]:
import plotly.express as px

In [ ]:
fig_imdb = px.histogram(
    df, 
    x='IMDb', 
    nbins=20,
    title='Quality Architecture: Density Distribution of IMDb Scores',
    labels={'IMDb': 'IMDb Rating Score'},
    color_discrete_sequence=['#8A2BE2'],
    marginal="box")
fig_imdb.update_layout(bargap=0.05, title_font_size=18)
fig_imdb.show()